In [0]:
df = spark.table("social_media.bronze.posts")
display(df)

In [0]:
from pyspark.sql.functions import *

clean_df = (
    df
    .withColumn(
        "clean_text",
        regexp_replace(col("content"), "[^a-zA-Z0-9 ]", "")
    )
)

In [0]:
clean_df.write \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("social_media.silver.cleaned_posts")

In [0]:
display(spark.table("social_media.silver.cleaned_posts"))

## Read Bronze Data

In [0]:
from pyspark.sql.functions import *

bronze_df=spark.table('social_media.bronze.posts')

In [0]:
display(bronze_df)

#### Clean Text

In [0]:
silver_df=bronze_df.withColumn('clean_text',regexp_replace(col('content'),"https//S+",""))
silver_df=silver_df.withColumn('clean_text',regexp_replace(col('clean_text'),"[^a-zA-Z0-9#@ ]",""))

#### Extract HashTags

In [0]:
silver_df=silver_df.withColumn('hashtags',split(regexp_replace(regexp_extract(col('content'),"(#\\w+.*)",0),"#","")," "))

#### Extract Mentions


In [0]:
silver_df=silver_df.withColumn("mentions",regexp_extract(col('content'),"@\\w+",0))

#### Add Word Count

In [0]:
silver_df=silver_df.withColumn('word_count',size(split(col('clean_text')," ")))

#### Language Detection

In [0]:
%pip install langdetect

In [0]:
from langdetect import detect
from pyspark.sql.types import StringType

def detect_lang(text):
    try:
        return detect(text)
    except:
        return "unknown"

lang_udf = udf(detect_lang, StringType())

silver_df = silver_df.withColumn(
    "language",
    lang_udf(col("clean_text"))
)

#### Add Timestamp 

In [0]:
silver_df = silver_df.withColumn(
    "processed_at",
    current_timestamp()
)

In [0]:
m

#### Remove Duplicate


In [0]:
silver_df = silver_df.dropDuplicates(["post_id"])

#### Filter Bad Records

In [0]:
silver_df = silver_df.filter(
    length(col("clean_text")) > 5
)

### Write To Silver


In [0]:
silver_df.write.mode('overwrite').option('overwriteSchema','true').saveAsTable("social_media.silver.cleaned_posts")